# 1  定义工具

In [4]:
from langchain_tavily import TavilySearch
import os
import dotenv
dotenv.load_dotenv()

search  = TavilySearch(tavily_api_key=os.environ['TAVILY_API_KEY'],
    max_results=5,
    topic="general"
)
# 查询 Tavily 搜索 API
# search = TavilySearchResults(tavily_api_key=os.getenv("TAVILY_API_KEY"),max_results=1)
# 执行查询
res = search.invoke("今天上海天气怎么样")
print(res)

{'query': '今天上海天气怎么样', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://tianqi.moji.com/today/china/shanghai/shanghai', 'title': '今天上海市天气_今日天气预报', 'content': '首页 天气 下载 资讯 关于墨迹. 2025年12月28日 乙巳[蛇]年 十一月初九. ## 今天实况. ## 今天白天. ## 今天夜间. ## 15天预报. ## 今天适合穿. 公司地址：北京市朝阳区来广营东路融新科技中心C座15层 联系电话：400-880-0599.', 'score': 0.9991768, 'raw_content': None}, {'url': 'https://www.nmc.cn/publish/forecast/ASH/shanghai.html', 'title': '上海-天气预报', 'content': '北京 18.3℃ 西南风 微风. 发布时间：12-27 12:00. 国家气象中心 版权所有 Copyright©2009-2025. 制作维护：国家气象中心预报系统开放实验室 地址：北京市中关村南大街46号 邮编：100081.', 'score': 0.9971771, 'raw_content': None}, {'url': 'https://weathernew.pae.baidu.com/weathernew/pc?query=%E4%B8%8A%E6%B5%B7%E5%A4%A9%E6%B0%94&srcid=4982&forecast=long_day_forecast', 'title': '上海', 'content': '7°C4°C1°C 15天天气预报 * 昨天 * 明天 * 周日 * 周一 * 周日 * 周一 未来39天天气预报 * 周日 * 周一 2 ~ 7°C 4 ~ 11°C 多云 6 ~ 15°C 6 ~ 15°C 北风3级 7 ~ 13°C 多云 北风1级 8 ~ 14°C 小雨 东风1级 8 ~ 12°C 5 ~ 11°C 3 ~ 9°C 4 ~ 12°C 多云 4 ~ 11°C 多云 4 

# 2 定义Retriever

In [3]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import dotenv
dotenv.load_dotenv()

# 1. 提供一个大模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

embedding_model = OpenAIEmbeddings()

# 2.加载HTML内容为一个文档对象
loader = WebBaseLoader("https://zh.wikipedia.org/wiki/%E7%8C%AB")
docs = loader.load()
#print(docs)

# 3.分割文档
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

documents = splitter.split_documents(docs)

# 4.向量化 得到向量数据库对象
vector = FAISS.from_documents(documents, embedding_model)

# 5.创建检索器
retriever = vector.as_retriever()

# 测试检索结果
# print(retriever.invoke("猫的特征")[0])

page_content='感官[编辑]
貓的感官適於狩獵。在哺乳類動物中，貓的聽覺、視覺、嗅覺、味覺、觸覺極敏銳。

視覺[编辑]
貓的瞳孔能縮得如線般狹小
貓的瞳孔變化
貓在晝間視覺縱不及人類，夜視能力與追蹤視覺上之活動物件卻相當出色，夜視能力是人類的六倍，雖然綜合色彩計算整體視覺系數則僅及人類的十分之一。貓的眼睛具有微光觀察能力，即使只有微弱月光都可分辨物件，有關光線入貓眼後可放大40至50倍處理，令貓具有夜間活動的能力，即使在黑暗的地下室貓咪依然能活動自如。強光下，貓會將瞳孔縮得如線般狹小，以減少對視網膜的傷害，但視野會因而縮窄。由於貓眼具備高幀與高分辨率視覺，故此電視機極微細之動靜對貓而言亦成逐格跳躍之畫面。[51]
另外，不只是光線會影響貓的瞳孔，感情也會。一般而言貓放鬆的時候瞳孔會縮小，而緊張時會放大。
貓的視網膜背面有一層藍綠色如熒光一般的薄膜（Tapetum Lucidum），可增加在暗處的視力。閃光中，貓眼能呈現各式各樣顔色。如同多數食肉動物，貓眼長在臉上朝正前方，賦予其遼闊的視野，單一貓眼視野為100度，雙眼視野為285度，加上頸旋靈活，總計視野比人類雙眼僅100度靈活得多。不過，貓僅能聚焦其前30厘米至3米之物件，相對人類而言屬大近視。
貓對三原色的辨識力很差，相對人類而言，貓有黃藍色盲且紅綠色辨成灰黃色，與狗相似。關於貓的夜視能力，生物學家發現牛磺酸對貓的視力起了很大作用，貓本身不能合成牛磺酸，必須由外攝取。缺乏牛磺酸，會使視力及夜視能力變差，影響夜間活動。據生物學家稱，貓經常捕鼠，是因老鼠體內含牛磺酸。[52][53]
紫外線可以通過貓的晶状体。
當四周光線微弱，貓會用感覺毛來改善行動力與感知能力。感覺毛主要分布於鼻子兩側、下巴、雙眼上方、兩頰也有數根。感覺毛可感受非常微弱的空氣波動，視野不清時也能協助辨識阻礙位置。鬍鬚尖端與雙耳連成一線，恰是身體能通過障礙的最小範圍，故貓可在黑夜中快速判斷地形能否通過。
貓有第三眼瞼，當貓眼睑張開時，眨眼時第三眼瞼會從旁稍微遮蓋眼睛。若貓生病，或是睡眠，笑著，此眼皮會縮回一部分。若貓長時間嶄露第三眼瞼，表示它的健康有問題。' metadata={'source': 'https://zh.wikipedia.org/wiki/%E7%8C%AB', 'title': '猫 - 维基百科，自由的百科全书'

# 3 创建工具、工具集

In [5]:
from langchain.tools.retriever import create_retriever_tool

# 创建一个工具来检索文档
retriever_tool = create_retriever_tool(
    retriever=retriever,
    name="wiki_search",
    description="搜索维基百科",
)

# 构建工具集
tools = [search, retriever_tool]

# 4 语言模型调用工具

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# 获取大模型
model = ChatOpenAI(model="gpt-4o-mini")

# 模型绑定工具
model_with_tools = model.bind_tools(tools)

# 根据输入自动调用工具
messages = [HumanMessage(content="今天上海天气怎么样")]
response = model_with_tools.invoke(messages)
print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")


ContentString: 
ToolCalls: [{'name': 'tavily_search_results_json', 'args': {'query': '今天上海天气'}, 'id': 'call_0ZwhabhDrd4N41HeRvzNCVW1', 'type': 'tool_call'}]


# 5 创建Agent程序(使用通用方式)

In [7]:
from langchain import hub
prompt = hub.pull("hwchase17/openai-functions-agent")

print(prompt.messages)

D:\developTools\miniconda3\envs\pyth310\lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}), MessagesPlaceholder(variable_name='chat_history', optional=True), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}), MessagesPlaceholder(variable_name='agent_scratchpad')]


In [8]:
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor

# 创建Agent对象
agent = create_tool_calling_agent(model, tools, prompt)

# 创建AgentExecutor对象
agent_executor = AgentExecutor(agent=agent, tools=tools,verbose=True)

# 6 运行Agent

In [9]:
print(agent_executor.invoke({"input": "猫的特征"}))



> Entering new AgentExecutor chain...

Invoking: `wiki_search` with `{'query': '猫的特征'}`


感官[编辑]
貓的感官適於狩獵。在哺乳類動物中，貓的聽覺、視覺、嗅覺、味覺、觸覺極敏銳。

視覺[编辑]
貓的瞳孔能縮得如線般狹小
貓的瞳孔變化
貓在晝間視覺縱不及人類，夜視能力與追蹤視覺上之活動物件卻相當出色，夜視能力是人類的六倍，雖然綜合色彩計算整體視覺系數則僅及人類的十分之一。貓的眼睛具有微光觀察能力，即使只有微弱月光都可分辨物件，有關光線入貓眼後可放大40至50倍處理，令貓具有夜間活動的能力，即使在黑暗的地下室貓咪依然能活動自如。強光下，貓會將瞳孔縮得如線般狹小，以減少對視網膜的傷害，但視野會因而縮窄。由於貓眼具備高幀與高分辨率視覺，故此電視機極微細之動靜對貓而言亦成逐格跳躍之畫面。[51]
另外，不只是光線會影響貓的瞳孔，感情也會。一般而言貓放鬆的時候瞳孔會縮小，而緊張時會放大。
貓的視網膜背面有一層藍綠色如熒光一般的薄膜（Tapetum Lucidum），可增加在暗處的視力。閃光中，貓眼能呈現各式各樣顔色。如同多數食肉動物，貓眼長在臉上朝正前方，賦予其遼闊的視野，單一貓眼視野為100度，雙眼視野為285度，加上頸旋靈活，總計視野比人類雙眼僅100度靈活得多。不過，貓僅能聚焦其前30厘米至3米之物件，相對人類而言屬大近視。
貓對三原色的辨識力很差，相對人類而言，貓有黃藍色盲且紅綠色辨成灰黃色，與狗相似。關於貓的夜視能力，生物學家發現牛磺酸對貓的視力起了很大作用，貓本身不能合成牛磺酸，必須由外攝取。缺乏牛磺酸，會使視力及夜視能力變差，影響夜間活動。據生物學家稱，貓經常捕鼠，是因老鼠體內含牛磺酸。[52][53]
紫外線可以通過貓的晶状体。
當四周光線微弱，貓會用感覺毛來改善行動力與感知能力。感覺毛主要分布於鼻子兩側、下巴、雙眼上方、兩頰也有數根。感覺毛可感受非常微弱的空氣波動，視野不清時也能協助辨識阻礙位置。鬍鬚尖端與雙耳連成一線，恰是身體能通過障礙的最小範圍，故貓可在黑夜中快速判斷地形能否通過。
貓有第三眼瞼，當貓眼睑張開時，眨眼時第三眼瞼會從旁稍微遮蓋眼睛。若貓生病，或是睡眠，笑著，此眼皮會縮回一部分。若貓長時間嶄露第三眼瞼，表示它的健康有問題。

貓爪[编辑]
貓的爪子尖

In [10]:
print(agent_executor.invoke({"input": "今天上海天气怎么样"}))



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': '上海天气'}`


[{'title': '上海-天气预报', 'url': 'https://www.nmc.cn/publish/forecast/ASH/shanghai.html', 'content': '土壤水分监测\n       农业干旱综合监测\n       关键农时农事\n       农业气象周报\n       农业气象月报\n       生态气象监测评估\n       农业气象专报\n       作物发育期监测\n       农业气象灾害风险预警\n       国外农业气象月报\n\n   数值预报\n\n       CMA全球天气模式\n       CMA全球集合模式\n       CMA区域模式\n       CMA区域集合模式\n       CMA台风模式\n       海浪模式\n\n1.    当前位置：首页\n2.   上海市\n3.   上海天气预报\n\n省份：城市：\n\n17:15更新\n\nImage 4\n\n日出05:35\n\n 上海 \n\n28.8℃\n\n日落18:09\n\n 降水量 \n\n0mm\n\n西南风\n\n微风\n\n 相对湿度 \n\n72%\n\n 体感温度 \n\n33.3℃\n\n空气质量：优\n\n舒适度：暖，不舒适\n\n 雷达图 \n\nImage 5\n\n24小时预报7天预报10天预报11-30天预报\n\n 发布时间：09-08 20:00 \n\n 09/08 \n\n周一 \n\n 27℃', 'score': 0.755078}]今天上海的天气为28.8℃，微风（西南风），相对湿度72%。体感温度约为33.3℃，没有降水，空气质量良好。总体上天气较为舒适，但由于体感温度较高，可能会感觉不够舒适。

详细天气预报可以查看 [这里](https://www.nmc.cn/publish/forecast/ASH/shanghai.html)。

> Finished chain.
{'input': '今天上海天气怎么样', 'output': '今天上海

# 7 添加记忆

In [11]:
from langchain_community.chat_message_histories import ChatMessageHistory

from langchain_core.chat_history import BaseChatMessageHistory

from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

# 调取指定session_id对应的memory
def get_session_history(session_id: str) -> BaseChatMessageHistory:

    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    return store[session_id]

agent_with_chat_history = RunnableWithMessageHistory(
    runnable=agent_executor,
    get_session_history=get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

response = agent_with_chat_history.invoke(
    {"input": "Hi，我的名字是Cyber"},
    config={"configurable": {"session_id": "123"}},
)

print(response)



> Entering new AgentExecutor chain...
你好，Cyber！很高兴见到你。有什么我可以帮助你的吗？

> Finished chain.
{'input': 'Hi，我的名字是Cyber', 'chat_history': [], 'output': '你好，Cyber！很高兴见到你。有什么我可以帮助你的吗？'}


In [12]:
response = agent_with_chat_history.invoke(
    {"input": "我叫什么名字?"},
    config={"configurable": {"session_id": "123"}},
)

print(response)



> Entering new AgentExecutor chain...
你叫Cyber。有什么特别的事情想和我分享吗？

> Finished chain.
{'input': '我叫什么名字?', 'chat_history': [HumanMessage(content='Hi，我的名字是Cyber', additional_kwargs={}, response_metadata={}), AIMessage(content='你好，Cyber！很高兴见到你。有什么我可以帮助你的吗？', additional_kwargs={}, response_metadata={})], 'output': '你叫Cyber。有什么特别的事情想和我分享吗？'}


In [13]:
response = agent_with_chat_history.invoke(
    {"input": "我叫什么名字?"},
    config={"configurable": {"session_id": "4566"}},
)
print(response)



> Entering new AgentExecutor chain...
抱歉，我无法得知您的名字。如果您愿意，可以告诉我您的名字！

> Finished chain.
{'input': '我叫什么名字?', 'chat_history': [], 'output': '抱歉，我无法得知您的名字。如果您愿意，可以告诉我您的名字！'}
